# ArmyOfSafeguards — ShieldGemma teacher + meta (Google Drive)

1. Runtime → **Change runtime type** → **GPU** (recommended).
2. Colab **Secrets** (`🔑`) → add secret name **`HF_TOKEN`** (Hugging Face read token; gated models/datasets).
3. Run cells top to bottom. Outputs go to `DRIVE_ROOT/AOS_outputs/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

# Change this if your Drive layout differs
DRIVE_ROOT = Path("/content/drive/MyDrive")
WORKSPACE = DRIVE_ROOT / "AOS_workspace"
OUT_DIR = DRIVE_ROOT / "AOS_outputs"

WORKSPACE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORKSPACE / "ArmyOfSafeguards"
print("WORKSPACE:", WORKSPACE)
print("OUT_DIR:", OUT_DIR)
print("REPO_DIR:", REPO_DIR)

In [ ]:
import os
import subprocess
import sys

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(
        ["git", "clone", "https://github.com/SohamNagi/ArmyOfSafeguards.git", str(REPO_DIR)],
        cwd=str(WORKSPACE),
    )
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull"], cwd=str(REPO_DIR))

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Set Colab Secret HF_TOKEN (🔑). If using a notebook copy without userdata, set os.environ['HF_TOKEN'] manually."
    ) from e

assert os.environ.get("HF_TOKEN", "").strip(), "HF_TOKEN is empty"

In [ ]:
# Optional: silence HF symlink warning on Colab (harmless either way)
import os

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

**Custom manifest:** This notebook `git clone`s the public repo. If you edited `training/teacher_dataset/manifest.meta_hf_labels.json` only on your PC, upload it in Colab (Files sidebar) to the same path under `.../ArmyOfSafeguards/training/teacher_dataset/`, or paste your JSON there, **before** running the `label_manifest.py` cell.

In [ ]:
# Teacher labels + expert features → meta-ready JSONL on Drive-backed repo copy
!python training/teacher_dataset/label_manifest.py \
  --manifest training/teacher_dataset/manifest.meta_hf_labels.json \
  --teacher shieldgemma \
  --device cuda \
  --threshold 0.5 \
  --require-expert-outputs \
  --out training/meta/teacher_all_for_meta.jsonl

In [ ]:
# Train unified meta policy (logistic regression over expert outputs)
!python -m meta_classifier.train_meta \
  --data training/meta/teacher_all_for_meta.jsonl \
  --n-folds 5 \
  --group-field source \
  --calibrate temperature \
  --out meta_classifier/artifacts/meta_lr.json

In [ ]:
import shutil
from pathlib import Path

src_jsonl = REPO_DIR / "training/meta/teacher_all_for_meta.jsonl"
src_model = REPO_DIR / "meta_classifier/artifacts/meta_lr.json"

for src in (src_jsonl, src_model):
    if src.exists():
        dst = OUT_DIR / src.name
        shutil.copy2(src, dst)
        print("copied:", dst)
    else:
        print("missing:", src)